# Dry-run read-only del plan de migracion Drive -> GCS

## Objetivo y garantias de seguridad

Este notebook lee el plan de migracion generado por el notebook 38, revalida que los archivos fuente sigan disponibles en Google Drive, autentica una sesion interactiva de Colab contra Google Cloud e inspecciona el estado actual del bucket de artifacts en modo estrictamente read-only.

Garantias:

- No sube objetos.
- No modifica objetos existentes.
- No copia, reemplaza ni elimina contenido remoto.
- No crea buckets, credenciales, claves ni recursos de infraestructura.
- No modifica los manifests originales.
- La escritura local de reportes esta desactivada por defecto y, si se activa manualmente, queda limitada a un directorio de resultados en Drive.

Una futura fase de carga se implementara recien despues de revisar este dry-run.

## Configuracion

Los valores congelados corresponden al plan validado en Colab. `DRY_RUN` debe ser exactamente `True`; cualquier otro valor aborta la ejecucion.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import tempfile
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import pandas as pd

DRY_RUN = True
CHECK_GCS_STATE = True
WRITE_DRY_RUN_REPORT = True

if DRY_RUN is not True:
    raise RuntimeError("DRY_RUN debe ser exactamente True. Abortando para evitar cualquier ejecucion insegura.")

PROJECT_ID = "pfi-asplanatti-fabrello-v1"
BUCKET_NAME = "pfi-rm-lumbar-artifacts-2026-ef"
BUCKET_URI_PREFIX = f"gs://{BUCKET_NAME}/"
GCS_PREFIX = "datasets/"
ALLOWED_ZERO_BYTE_PLACEHOLDERS = {GCS_PREFIX}
EXPECTED_BUCKET_LOCATION = "US-CENTRAL1"
EXPECTED_BUCKET_STORAGE_CLASS = "STANDARD"

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"
PLAN_DIR = PFI_ROOT / "results" / "GCS_dataset_migration_plan"
REPORT_DIR = PFI_ROOT / "results" / "GCS_dataset_upload_dry_run"

MANIFEST_CSV = PLAN_DIR / "gcs_upload_manifest.csv"
SUMMARY_CSV = PLAN_DIR / "gcs_upload_summary.csv"
VALIDATION_JSON = PLAN_DIR / "gcs_upload_validation.json"

EXPECTED_MANIFEST_ROWS = 2058
EXPECTED_TOTAL_BYTES = 3922288649
EXPECTED_TOTAL_GIB = 3.652915962971747
EXPECTED_COMPONENT_COUNTS = {
    "spider_image": 447,
    "spider_mask": 447,
    "axial_image": 610,
    "axial_mask": 552,
    "metadata_e5": 1,
    "metadata_e9": 1,
    "bootstrap_model": 0,
}
EXPECTED_MANIFEST_COLUMNS = [
    "component",
    "source_path",
    "source_root",
    "relative_path",
    "destination_uri",
    "suffix",
    "size_bytes",
    "modified_at",
    "sha256",
    "referenced_rows",
    "exists",
    "is_symlink",
    "status",
]
METADATA_COMPONENTS = {"metadata_e5", "metadata_e9"}
ALLOWED_REPORT_FILES = {"gcs_dry_run_summary.json", "gcs_dry_run_objects.csv"}

print(json.dumps({
    "DRY_RUN": DRY_RUN,
    "CHECK_GCS_STATE": CHECK_GCS_STATE,
    "WRITE_DRY_RUN_REPORT": WRITE_DRY_RUN_REPORT,
    "project": PROJECT_ID,
    "bucket": BUCKET_NAME,
    "plan_dir": str(PLAN_DIR),
}, indent=2))

{
  "DRY_RUN": true,
  "CHECK_GCS_STATE": true,
  "WRITE_DRY_RUN_REPORT": true,
  "project": "pfi-asplanatti-fabrello-v1",
  "bucket": "pfi-rm-lumbar-artifacts-2026-ef",
  "plan_dir": "/content/drive/MyDrive/PFI_MVP/results/GCS_dataset_migration_plan"
}


## Montaje de Drive

Reutiliza `/content/drive/MyDrive` si ya esta disponible. Si no esta montado, solicita el montaje normal de Colab sin forzar remontaje y sin limpiar mountpoints.

In [2]:
def ensure_drive_available() -> None:
    if MY_DRIVE.exists():
        print(f"Drive ya disponible: {MY_DRIVE}")
    else:
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError("Este notebook debe ejecutarse en Colab o con /content/drive/MyDrive ya disponible.") from exc
        drive.mount(str(DRIVE_ROOT))

    if not PFI_ROOT.exists():
        raise FileNotFoundError(f"No existe la raiz esperada de PFI: {PFI_ROOT}")
    print(f"PFI_ROOT OK: {PFI_ROOT}")

ensure_drive_available()

Drive ya disponible: /content/drive/MyDrive
PFI_ROOT OK: /content/drive/MyDrive/PFI_MVP


## Validacion del manifest

Carga los archivos del plan y valida conteos, columnas, estados, destinos, duplicados y restricciones de rutas antes de consultar GCS.

In [3]:
def require_readable_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Falta archivo requerido: {path}")
    if not path.is_file():
        raise ValueError(f"No es un archivo regular: {path}")
    with path.open("rb") as fh:
        fh.read(1)


def parse_bool_series(series: pd.Series, name: str) -> pd.Series:
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.strip().str.lower()
    allowed = {"true", "false"}
    bad = sorted(set(normalized) - allowed)
    if bad:
        raise ValueError(f"Columna booleana {name} contiene valores invalidos: {bad[:10]}")
    return normalized == "true"


def path_is_inside(path_text: str, root: Path) -> bool:
    try:
        path = Path(path_text)
        return path.is_absolute() and (path == root or root in path.parents)
    except Exception:
        return False


def destination_object_name(uri: str) -> str:
    if not isinstance(uri, str) or not uri.startswith(BUCKET_URI_PREFIX):
        raise ValueError(f"destination_uri fuera del bucket esperado: {uri}")
    object_name = uri[len(BUCKET_URI_PREFIX):]
    if not object_name:
        raise ValueError(f"destination_uri sin object name: {uri}")
    return object_name


def validate_plan_files() -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    for path in [MANIFEST_CSV, SUMMARY_CSV, VALIDATION_JSON]:
        require_readable_file(path)

    manifest = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    summary = pd.read_csv(SUMMARY_CSV, keep_default_na=False)
    with VALIDATION_JSON.open("r", encoding="utf-8") as fh:
        validation = json.load(fh)

    if list(manifest.columns) != EXPECTED_MANIFEST_COLUMNS:
        raise ValueError(f"Columnas inesperadas en manifest: {list(manifest.columns)}")
    if len(manifest) != EXPECTED_MANIFEST_ROWS:
        raise ValueError(f"Filas inesperadas: {len(manifest)} != {EXPECTED_MANIFEST_ROWS}")

    manifest["size_bytes"] = pd.to_numeric(manifest["size_bytes"], errors="raise").astype("int64")
    manifest["exists"] = parse_bool_series(manifest["exists"], "exists")
    manifest["is_symlink"] = parse_bool_series(manifest["is_symlink"], "is_symlink")

    total_bytes = int(manifest["size_bytes"].sum())
    if total_bytes != EXPECTED_TOTAL_BYTES:
        raise ValueError(f"Bytes inesperados: {total_bytes} != {EXPECTED_TOTAL_BYTES}")

    component_counts = manifest["component"].value_counts().to_dict()
    for component, expected in EXPECTED_COMPONENT_COUNTS.items():
        actual = int(component_counts.get(component, 0))
        if actual != expected:
            raise ValueError(f"Conteo inesperado para {component}: {actual} != {expected}")
    unexpected_components = sorted(set(component_counts) - set(EXPECTED_COMPONENT_COUNTS))
    if unexpected_components:
        raise ValueError(f"Componentes no esperados: {unexpected_components}")
    if (manifest["component"] == "bootstrap_model").any():
        raise ValueError("bootstrap_model no debe estar incluido en este plan congelado")

    bad_status = manifest[manifest["status"].isna() | (manifest["status"].astype(str) != "OK")]
    if not bad_status.empty:
        raise ValueError(f"Hay filas con status no OK: {bad_status[['component', 'source_path', 'status']].head(10).to_dict('records')}")
    if not bool(manifest["exists"].all()):
        raise ValueError("Hay filas con exists != True")
    if bool(manifest["is_symlink"].any()):
        raise ValueError("Hay filas marcadas como symlink")
    if bool((manifest["size_bytes"] <= 0).any()):
        raise ValueError("Hay filas con size_bytes <= 0")
    if manifest["destination_uri"].duplicated().any():
        dup = manifest.loc[manifest["destination_uri"].duplicated(keep=False), "destination_uri"].head(10).tolist()
        raise ValueError(f"destination_uri duplicados: {dup}")
    if manifest.duplicated(["component", "source_path"]).any():
        dup = manifest.loc[manifest.duplicated(["component", "source_path"], keep=False), ["component", "source_path"]].head(10).to_dict("records")
        raise ValueError(f"source_path duplicado dentro de componente: {dup}")
    if bool(manifest["relative_path"].astype(str).str.strip().eq("").any()):
        raise ValueError("Hay relative_path vacios")
    if bool(manifest["relative_path"].astype(str).str.contains("..", regex=False).any()):
        raise ValueError('Hay relative_path con ".."')
    if bool(manifest["suffix"].astype(str).str.lower().isin({".zip"}).any()):
        raise ValueError("Hay archivos ZIP incluidos")
    if not bool(manifest["destination_uri"].astype(str).str.startswith(BUCKET_URI_PREFIX).all()):
        raise ValueError("Hay destination_uri fuera del bucket esperado")
    if not bool(manifest["source_path"].map(lambda value: path_is_inside(str(value), PFI_ROOT)).all()):
        raise ValueError(f"Hay source_path fuera de {PFI_ROOT}")

    if validation.get("errors") != []:
        raise ValueError(f"validation.errors no esta vacio: {validation.get('errors')}")
    if validation.get("warnings") != []:
        raise ValueError(f"validation.warnings no esta vacio: {validation.get('warnings')}")
    if validation.get("duplicated_destinations") != []:
        raise ValueError(f"validation.duplicated_destinations no esta vacio: {validation.get('duplicated_destinations')}")

    print(json.dumps({
        "manifest_rows": len(manifest),
        "manifest_bytes": total_bytes,
        "manifest_gib": total_bytes / (1024 ** 3),
        "expected_gib": EXPECTED_TOTAL_GIB,
        "component_counts": {k: int(component_counts.get(k, 0)) for k in EXPECTED_COMPONENT_COUNTS},
    }, indent=2))
    return manifest, summary, validation

manifest_df, summary_df, validation_payload = validate_plan_files()
LOCAL_PLAN_READY = True

{
  "manifest_rows": 2058,
  "manifest_bytes": 3922288649,
  "manifest_gib": 3.652915962971747,
  "expected_gib": 3.652915962971747,
  "component_counts": {
    "spider_image": 447,
    "spider_mask": 447,
    "axial_image": 610,
    "axial_mask": 552,
    "metadata_e5": 1,
    "metadata_e9": 1,
    "bootstrap_model": 0
  }
}


## Revalidacion de fuentes

Verifica que cada archivo local del manifest exista, sea regular, no sea symlink y conserve exactamente el mismo tama?o. Solo los archivos de metadata E5/E9 se verifican con SHA-256 completo por streaming.

In [4]:
def sha256_stream(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def revalidate_source_files(manifest: pd.DataFrame) -> None:
    total = len(manifest)
    for index, row in enumerate(manifest.itertuples(index=False), start=1):
        path = Path(row.source_path)
        if not path.exists():
            raise FileNotFoundError(f"Fuente faltante: component={row.component} relative_path={row.relative_path}")
        if path.is_symlink():
            raise ValueError(f"Fuente symlink rechazada: component={row.component} relative_path={row.relative_path}")
        if not path.is_file():
            raise ValueError(f"Fuente no regular: component={row.component} relative_path={row.relative_path}")
        current_size = path.stat().st_size
        if current_size != int(row.size_bytes):
            raise ValueError(
                f"Size mismatch: component={row.component} relative_path={row.relative_path} "
                f"actual={current_size} manifest={int(row.size_bytes)}"
            )
        if row.component in METADATA_COMPONENTS:
            actual_sha = sha256_stream(path)
            expected_sha = str(row.sha256).strip()
            if actual_sha != expected_sha:
                raise ValueError(
                    f"SHA-256 mismatch en {row.component}: relative_path={row.relative_path} "
                    f"actual={actual_sha} manifest={expected_sha}"
                )
        if index % 250 == 0 or index == total:
            print(f"Fuentes revalidadas: {index}/{total}")

revalidate_source_files(manifest_df)
SOURCE_FILES_READY = True

Fuentes revalidadas: 250/2058
Fuentes revalidadas: 500/2058
Fuentes revalidadas: 750/2058
Fuentes revalidadas: 1000/2058
Fuentes revalidadas: 1250/2058
Fuentes revalidadas: 1500/2058
Fuentes revalidadas: 1750/2058
Fuentes revalidadas: 2000/2058
Fuentes revalidadas: 2058/2058


## Autenticacion GCP

Usa la autenticacion interactiva de Colab y las librerias oficiales de Google. No usa archivos de credenciales ni imprime informacion sensible.

In [5]:
def authenticate_gcp_readonly():
    try:
        from google.colab import auth
    except ImportError as exc:
        raise RuntimeError("La autenticacion interactiva requiere Google Colab.") from exc

    try:
        import google.auth
        from google.cloud import storage
    except ImportError as exc:
        raise RuntimeError("Falta google-cloud-storage/google-auth en el entorno. Instalar manualmente antes de ejecutar esta celda.") from exc

    auth.authenticate_user()
    credentials, active_project = google.auth.default()
    client = storage.Client(project=PROJECT_ID, credentials=credentials)
    print(json.dumps({"client_project": PROJECT_ID, "auth_project_detected": active_project}, indent=2))
    return client

storage_client = authenticate_gcp_readonly()

{
  "client_project": "pfi-asplanatti-fabrello-v1",
  "auth_project_detected": ""
}


## Inspeccion read-only del bucket

Obtiene metadata del bucket existente y lista una sola vez los objetos bajo `datasets/`. Si el bucket no existe o su metadata no coincide, aborta.

In [6]:
def inspect_bucket_readonly(client) -> tuple[Any, dict[str, dict[str, Any]]]:
    if CHECK_GCS_STATE is not True:
        raise RuntimeError("CHECK_GCS_STATE debe permanecer True para este dry-run.")

    bucket = client.bucket(BUCKET_NAME)
    bucket.reload(client=client)

    location = str(bucket.location or "").upper()
    storage_class = str(bucket.storage_class or "").upper()
    if bucket.name != BUCKET_NAME:
        raise ValueError(f"Bucket inesperado: {bucket.name}")
    if location != EXPECTED_BUCKET_LOCATION:
        raise ValueError(f"Location inesperada: {location} != {EXPECTED_BUCKET_LOCATION}")
    if storage_class != EXPECTED_BUCKET_STORAGE_CLASS:
        raise ValueError(f"Storage class inesperada: {storage_class} != {EXPECTED_BUCKET_STORAGE_CLASS}")

    existing: dict[str, dict[str, Any]] = {}
    for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        existing[blob.name] = {
            "name": blob.name,
            "size": int(blob.size or 0),
            "crc32c": blob.crc32c,
            "md5_hash": blob.md5_hash,
            "generation": str(blob.generation) if blob.generation is not None else None,
            "updated": blob.updated.isoformat() if blob.updated is not None else None,
        }

    print(json.dumps({
        "bucket": bucket.name,
        "location": location,
        "storage_class": storage_class,
        "objects_under_datasets": len(existing),
    }, indent=2))
    return bucket, existing

bucket, existing_objects = inspect_bucket_readonly(storage_client)
BUCKET_METADATA_READY = True

{
  "bucket": "pfi-rm-lumbar-artifacts-2026-ef",
  "location": "US-CENTRAL1",
  "storage_class": "STANDARD",
  "objects_under_datasets": 1
}


## Clasificacion del dry-run

Clasifica cada destino planificado contra el inventario remoto ya listado. Un objeto con igual tama?o queda como igualdad no verificada, no como match seguro de contenido.

In [7]:
def classify_dry_run(manifest: pd.DataFrame, existing: dict[str, dict[str, Any]]) -> tuple[pd.DataFrame, dict[str, int]]:
    rows: list[dict[str, Any]] = []
    planned_objects: set[str] = set()

    for row in manifest.itertuples(index=False):
        object_name = destination_object_name(str(row.destination_uri))
        if not object_name.startswith(GCS_PREFIX):
            raise ValueError(f"Destino fuera de prefijo {GCS_PREFIX}: {object_name}")
        planned_objects.add(object_name)
        remote = existing.get(object_name)
        if remote is None:
            dry_status = "TO_UPLOAD"
            remote_size = None
        else:
            remote_size = int(remote["size"])
            if remote_size == int(row.size_bytes):
                dry_status = "EXISTS_SAME_SIZE_UNVERIFIED"
            else:
                dry_status = "CONFLICT_SIZE"
        rows.append({
            "component": row.component,
            "source_path": row.source_path,
            "relative_path": row.relative_path,
            "destination_uri": row.destination_uri,
            "object_name": object_name,
            "manifest_size_bytes": int(row.size_bytes),
            "remote_size_bytes": remote_size,
            "dry_run_status": dry_status,
        })

    unplanned = sorted(set(existing) - planned_objects)
    for object_name in unplanned:
        remote = existing[object_name]
        remote_size = int(remote["size"])
        if object_name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and remote_size == 0:
            dry_status = "ALLOWED_ZERO_BYTE_PLACEHOLDER"
            component = "ALLOWED_ZERO_BYTE_PLACEHOLDER"
        else:
            dry_status = "UNPLANNED_EXISTING"
            component = "UNPLANNED_EXISTING"
        rows.append({
            "component": component,
            "source_path": "",
            "relative_path": "",
            "destination_uri": BUCKET_URI_PREFIX + object_name,
            "object_name": object_name,
            "manifest_size_bytes": None,
            "remote_size_bytes": remote_size,
            "dry_run_status": dry_status,
        })

    objects_df = pd.DataFrame(rows)
    counts = objects_df["dry_run_status"].value_counts().to_dict()
    normalized_counts = {
        "TO_UPLOAD": int(counts.get("TO_UPLOAD", 0)),
        "EXISTS_SAME_SIZE_UNVERIFIED": int(counts.get("EXISTS_SAME_SIZE_UNVERIFIED", 0)),
        "CONFLICT_SIZE": int(counts.get("CONFLICT_SIZE", 0)),
        "UNPLANNED_EXISTING": int(counts.get("UNPLANNED_EXISTING", 0)),
        "ALLOWED_ZERO_BYTE_PLACEHOLDER": int(counts.get("ALLOWED_ZERO_BYTE_PLACEHOLDER", 0)),
    }
    print(json.dumps(normalized_counts, indent=2))
    return objects_df, normalized_counts

dry_run_objects_df, dry_run_counts = classify_dry_run(manifest_df, existing_objects)

{
  "TO_UPLOAD": 2058,
  "EXISTS_SAME_SIZE_UNVERIFIED": 0,
  "CONFLICT_SIZE": 0,
  "UNPLANNED_EXISTING": 0,
  "ALLOWED_ZERO_BYTE_PLACEHOLDER": 1
}


## Quality gate

El gate final solo queda listo si el plan local, las fuentes y el bucket coinciden con lo esperado, y el prefijo remoto no contiene destinos planificados ni objetos no planificados.

In [8]:
NO_EXISTING_PLANNED_OBJECTS = (
    dry_run_counts["EXISTS_SAME_SIZE_UNVERIFIED"] == 0
    and dry_run_counts["CONFLICT_SIZE"] == 0
)
NO_UNPLANNED_DATASET_OBJECTS = dry_run_counts["UNPLANNED_EXISTING"] == 0
GCS_DRY_RUN_READY = bool(
    DRY_RUN is True
    and LOCAL_PLAN_READY
    and SOURCE_FILES_READY
    and BUCKET_METADATA_READY
    and NO_EXISTING_PLANNED_OBJECTS
    and NO_UNPLANNED_DATASET_OBJECTS
    and dry_run_counts["CONFLICT_SIZE"] == 0
)

final_summary = {
    "DRY_RUN": DRY_RUN,
    "manifest_rows": int(len(manifest_df)),
    "manifest_bytes": int(manifest_df["size_bytes"].sum()),
    "to_upload": dry_run_counts["TO_UPLOAD"],
    "exists_same_size_unverified": dry_run_counts["EXISTS_SAME_SIZE_UNVERIFIED"],
    "conflict_size": dry_run_counts["CONFLICT_SIZE"],
    "unplanned_existing": dry_run_counts["UNPLANNED_EXISTING"],
    "allowed_zero_byte_placeholders": dry_run_counts["ALLOWED_ZERO_BYTE_PLACEHOLDER"],
    "LOCAL_PLAN_READY": bool(LOCAL_PLAN_READY),
    "SOURCE_FILES_READY": bool(SOURCE_FILES_READY),
    "BUCKET_METADATA_READY": bool(BUCKET_METADATA_READY),
    "NO_EXISTING_PLANNED_OBJECTS": bool(NO_EXISTING_PLANNED_OBJECTS),
    "NO_UNPLANNED_DATASET_OBJECTS": bool(NO_UNPLANNED_DATASET_OBJECTS),
    "GCS_DRY_RUN_READY": bool(GCS_DRY_RUN_READY),
}
print(json.dumps(final_summary, indent=2))

if not GCS_DRY_RUN_READY:
    raise RuntimeError("GCS_DRY_RUN_READY=False. Revisar conflictos, objetos existentes o metadata antes de habilitar una fase futura.")

{
  "DRY_RUN": true,
  "manifest_rows": 2058,
  "manifest_bytes": 3922288649,
  "to_upload": 2058,
  "exists_same_size_unverified": 0,
  "conflict_size": 0,
  "unplanned_existing": 0,
  "allowed_zero_byte_placeholders": 1,
  "LOCAL_PLAN_READY": true,
  "SOURCE_FILES_READY": true,
  "BUCKET_METADATA_READY": true,
  "NO_EXISTING_PLANNED_OBJECTS": true,
  "NO_UNPLANNED_DATASET_OBJECTS": true,
  "GCS_DRY_RUN_READY": true
}


## Reporte opcional

Por defecto no escribe nada. Si se activa manualmente el flag de reporte, solo guarda dos archivos permitidos dentro del directorio de resultados del dry-run, usando escritura atomica.

In [9]:
def atomic_write_text(path: Path, text: str) -> None:
    if path.parent != REPORT_DIR:
        raise ValueError(f"Directorio de reporte no permitido: {path.parent}")
    if path.name not in ALLOWED_REPORT_FILES:
        raise ValueError(f"Archivo de reporte no permitido: {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as fh:
            fh.write(text)
            fh.flush()
            os.fsync(fh.fileno())
        os.replace(tmp_name, path)
    finally:
        tmp_path = Path(tmp_name)
        if tmp_path.exists():
            tmp_path.unlink()


def dataframe_to_csv_text(df: pd.DataFrame) -> str:
    return df.to_csv(index=False, quoting=csv.QUOTE_MINIMAL)

if WRITE_DRY_RUN_REPORT:
    atomic_write_text(REPORT_DIR / "gcs_dry_run_summary.json", json.dumps(final_summary, indent=2, ensure_ascii=False) + "\n")
    atomic_write_text(REPORT_DIR / "gcs_dry_run_objects.csv", dataframe_to_csv_text(dry_run_objects_df))
    print(f"Reporte dry-run escrito en {REPORT_DIR}")
else:
    print("WRITE_DRY_RUN_REPORT=False; no se escribieron reportes.")

Reporte dry-run escrito en /content/drive/MyDrive/PFI_MVP/results/GCS_dataset_upload_dry_run


## Proxima fase

Revisar el JSON final y, si `GCS_DRY_RUN_READY=True`, crear un ticket separado para implementar una carga controlada. Este notebook permanece como evidencia read-only previa a cualquier transferencia.